In [3]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="keras")
import gdown
import zipfile

In [ ]:
data_url = "https://drive.google.com/file/d/1ix5O0WOFLBc3JyWceXdPkyOnUgzCxJgE/view?usp=sharing"
downloaded_file = gdown.download(data_url, output = None, quiet = False)
extract_folder = '.'
with zipfile.ZipFile(downloaded_file, 'r') as zip_ref:
        zip_ref.extractall(extract_folder)
os.remove(downloaded_file)

Downloading...
From (original): https://drive.google.com/uc?id=1ix5O0WOFLBc3JyWceXdPkyOnUgzCxJgE
From (redirected): https://drive.google.com/uc?id=1ix5O0WOFLBc3JyWceXdPkyOnUgzCxJgE&confirm=t&uuid=1cbe9afd-0f80-4383-bfd4-f11e9ebbb9a0
To: /home/norbert/Projects/computer_vision_oly/classification/watermark_detection/dk-dataset.zip
100%|██████████| 210M/210M [01:19<00:00, 2.65MB/s] 


In [5]:
TRAIN_DIR = 'dataset/train'
TEST_DIR = 'dataset/test'

In [6]:
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_generator = datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

class_indices = train_generator.class_indices
print(class_indices)

Found 6866 images belonging to 2 classes.
Found 1716 images belonging to 2 classes.
{'negative': 0, 'positive': 1}


In [4]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
base_model.trainable = False 

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.2)(x)
x = Dense(64, activation='relu')(x)
predictions = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer=Adam(learning_rate=0.001), 
              loss='binary_crossentropy', 
              metrics=['accuracy'])
model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5
)

I0000 00:00:1789814042.015416 3091637 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4035 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Epoch 1/5


I0000 00:00:1789814043.408123 3091637 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1789814045.785793 3092822 service.cc:153] XLA service 0x7e7c5c0341f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1789814045.785812 3092822 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.22.0)
I0000 00:00:1789814045.848836 3092822 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1789814046.399235 3092822 cuda_dnn.cc:461] Loaded cuDNN version 92200
I0000 00:00:1789814046.451087 3092822 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_8108__.115
E0000 00:00:1789814050.124845 3092822 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup e

  5/215 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - accuracy: 0.4961 - loss: 0.8346 

I0000 00:00:1789814053.573086 3092822 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


122/215 ━━━━━━━━━━━━━━━━━━━━ 3s 36ms/step - accuracy: 0.5811 - loss: 0.7125

I0000 00:00:1789814058.608954 3092824 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_8108__.115
E0000 00:00:1789814060.725004 3092824 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


215/215 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.6053 - loss: 0.6835

E0000 00:00:1789814073.495936 3092825 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


215/215 ━━━━━━━━━━━━━━━━━━━━ 34s 111ms/step - accuracy: 0.6512 - loss: 0.6298 - val_accuracy: 0.6958 - val_loss: 0.5831
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 10s 49ms/step - accuracy: 0.7343 - loss: 0.5291 - val_accuracy: 0.7279 - val_loss: 0.5455
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 9s 43ms/step - accuracy: 0.7760 - loss: 0.4749 - val_accuracy: 0.7255 - val_loss: 0.5415
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 9s 42ms/step - accuracy: 0.8070 - loss: 0.4307 - val_accuracy: 0.7372 - val_loss: 0.5248
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 9s 42ms/step - accuracy: 0.8363 - loss: 0.3755 - val_accuracy: 0.7354 - val_loss: 0.5430


In [5]:
test_filenames = [f for f in os.listdir(TEST_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
test_df = pd.DataFrame({'name': test_filenames})

test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=TEST_DIR,
    x_col='name',
    y_col=None,
    target_size=(128, 128),
    batch_size=32,
    class_mode=None,
    shuffle=False
)

preds = model.predict(test_generator)

preds_binary = (preds > 0.5).astype(int).flatten()

submission = pd.DataFrame({
    'name': test_df['name'], 
    'predicted': preds_binary
})

submission.to_csv('output.csv', index=False)

print(submission.head())

Found 1103 validated image filenames.
34/35 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step

E0000 00:00:1789814134.897604 3092822 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


35/35 ━━━━━━━━━━━━━━━━━━━━ 9s 201ms/step
                                       name  predicted
0  2f6f7f0c-038b-44d3-b1af-af98fb65d17b.jpg          1
1  3b323266-18ab-4316-bb0b-ce06ce9bfa68.jpg          0
2  c010a0a5-b59c-4b93-ba43-c50ea40cfa76.jpg          0
3  48fd3d49-075a-4027-b04e-151c22294847.jpg          0
4  c76162c3-791c-42b8-b44c-fe18243ca2b9.jpg          0
